# 03. Modellwahl und Evaluation auf IMDb

**Leitfrage:** Wie wählen wir mit bekannten Filmreviews eine kNN-Konfiguration für neue Texte?

**Demonstration:** [IMDb-Dateien laden und einen ersten Lauf auswerten](#imdb)

**Arbeitsauftrag:** [Einstellungen auf der Validierung vergleichen](#model-selection), eine Prognose notieren und [am Testset prüfen](#final-test).

In [ ]:
%pip install scikit-learn==1.9.1

<a id="imdb"></a>

## 1. IMDb-Daten laden und den ersten Lauf auswerten

Die Kursauswahl enthält 700 Filmreviews. **500 Trainingsreviews** liefern Vokabular und gelabelte Nachbarn. **100 Validierungsreviews** zeigen, wie oft das Modell bei neuen Texten richtigliegt. Weitere **100 Testreviews** bleiben für die abschließende Prüfung zurück.

### 1.1 Reviewdateien laden

Jede `.txt`-Datei enthält eine vollständige Review. Der übergeordnete Ordner gibt ihr Urteil an: `pos` für positiv, `neg` für negativ. Die Nummer im Dateinamen legt die Reihenfolge fest. Öffne zuerst eine Trainingsdatei. [Kursdaten und Herkunft](data/imdb/README.md)

In [ ]:
from pathlib import Path

data_dir = Path('data/imdb')
train_folder = data_dir / 'train'
paths = sorted(train_folder.glob('*/*.txt'), key=lambda path: path.name)
first_path = paths[0]
print(first_path.relative_to(data_dir))

Wie liest du den Text aus `first_path`? Speichere ihn in `first_text` und zeige die ersten 180 Zeichen an.

In [ ]:
# TODO: Read first_path as UTF-8 and print the first 180 characters.

Dasselbe Muster funktioniert für einen ganzen Ordner. Die Schleife hängt für jede Datei den Text und das zugehörige Urteil an derselben Position an. Nur der Text wird später vektorisiert. Die Funktion gibt auch die zugehörigen Dateipfade zurück. Das Unterstrichzeichen `_` steht beim Laden von Training und Validierung für den Rückgabewert, den wir erst für den Testexport brauchen.

In [ ]:
def load_reviews(folder: Path) -> tuple[list[str], list[str], list[Path]]:
    paths = sorted(folder.glob('*/*.txt'), key=lambda path: path.name)
    texts = []
    labels = []
    for path in paths:
        texts.append(path.read_text(encoding='utf-8'))
        labels.append(path.parent.name)
    return texts, labels, paths

In [ ]:
train_texts, train_labels, _ = load_reviews(train_folder)
validation_texts, validation_labels, _ = load_reviews(data_dir / 'validation')
print('Training:', len(train_texts), '| Validation:', len(validation_texts))

### 1.2 Wortspalten festlegen

Ein neuer `CountVectorizer` lernt das Vokabular an den 500 Trainingsreviews. Für die Validierungsreviews verwendet er genau diese Spalten. [CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
train_vectors = vectorizer.fit_transform(train_texts)

Die Wortspalten stehen jetzt fest. Mit welcher Methode erhältst du für die Validierungsreviews dieselben Spalten? Speichere das Ergebnis in `validation_vectors`.

In [ ]:
# TODO: Vectorize validation_texts with the fitted vectorizer.

In [ ]:
print(train_vectors.shape, validation_vectors.shape)

### 1.3 Drei Nachbarn abstimmen lassen

Wie im kleinen Korpus verwenden wir drei Nachbarn und die Kosinusdistanz. Wir betrachten zuerst die Nachbarn der ersten Validierungsreview. [KNeighborsClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

imdb_model = KNeighborsClassifier(n_neighbors=3, metric='cosine')
imdb_model.fit(train_vectors, train_labels)

In [ ]:
example = validation_vectors[:1]
distances, indices = imdb_model.kneighbors(example)
print('Review:', validation_texts[0][:180])
for distance, index in zip(distances[0], indices[0]):
    print(f'{distance:.3f}', train_labels[index], train_texts[index][:100])

Welches Label gewinnt die Abstimmung? Vergleiche deinen Tipp mit der Modellvorhersage und dem bekannten Label der ersten Validierungsreview.

In [ ]:
predictions = imdb_model.predict(validation_vectors)
print('First prediction:', predictions[0], '| Actual:', validation_labels[0])

### 1.4 Trefferquote berechnen

Wie viele der 100 Vorhersagen stimmen mit dem bekannten Label überein? Der Anteil richtiger Vorhersagen heißt Accuracy.

In [ ]:
correct = sum(predictions == validation_labels)
print('Validation correct:', correct, '/', len(validation_labels))
print(f'Accuracy: {correct / len(validation_labels):.0%}')

<a id='model-selection'></a>

## 2. Hyperparameter auf der Validierung wählen

### 2.1 Einstellungen ausprobieren

Welche Einstellung liefert auf den 100 Validierungsreviews die höchste Accuracy? Wähle anhand der Dokumentation einen Parameter, etwa Nachbarzahl, Distanzmaß oder Stimmgewichtung. Ändere zunächst nur diesen Wert. Führe die nächsten drei Zellen für jeden Versuch erneut aus und notiere Einstellung und Trefferquote. Trainingsdaten und Wortspalten bleiben bei allen Versuchen gleich. [Parameter des KNeighborsClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html)

In [ ]:
# TODO: Try different parameter values.
classifier = KNeighborsClassifier(
    n_neighbors=3,
    metric='cosine',
    weights='uniform',
)

In [ ]:
classifier.fit(train_vectors, train_labels)
validation_predictions = classifier.predict(validation_vectors)

In [ ]:
from sklearn.metrics import accuracy_score

validation_accuracy = accuracy_score(validation_labels, validation_predictions)
print(f'Validation accuracy: {validation_accuracy:.0%}')

### 2.2 Modell festlegen und Prognose notieren

Stelle deine beste Konfiguration oben wieder ein und führe die drei Zellen aus 2.1 noch einmal aus. Mit diesem trainierten Modell geht es in den Test.

**Meine Prognose:** Ich habe … gewählt und auf der Validierung … % erreicht. Auf 100 neuen Reviews erwarte ich … Treffer, weil …

<a id='final-test'></a>

## 3. Wie gut war deine Prognose?

Öffne jetzt das Testset. Der trainierte Vektorisierer und der gewählte Klassifikator werden direkt auf die neuen Reviews angewendet.

In [ ]:
test_texts, test_labels, test_paths = load_reviews(data_dir / 'test')
test_vectors = vectorizer.transform(test_texts)
test_predictions = classifier.predict(test_vectors)

In [ ]:
test_accuracy = accuracy_score(test_labels, test_predictions)
print(f'Validation accuracy: {validation_accuracy:.0%}')
print(f'Test accuracy: {test_accuracy:.0%}')

Vergleiche deine Prognose mit der gemessenen Trefferquote. Wie groß ist der Abstand zwischen Validierung und Test in Prozentpunkten? Warum brauchten wir für diesen Vergleich die zurückgehaltenen Reviews?

### Testvorhersagen für Notebook 04 speichern

Lege für jede Testreview ein Wörterbuch an: `file_name` ist der Name der eingelesenen Textdatei, `actual` ihr bekanntes Urteil und `predicted` die Vorhersage deines Modells. Der Export schreibt die Wörterbücher als Zeilen in `my_test_predictions.csv`. In [Notebook 04](04_evaluation_metrics.ipynb) kannst du damit die Fehler deines Modells zählen.

Die mitgelieferte [reference_test_predictions.csv](data/imdb/reference_test_predictions.csv) wurde auf dieselbe Weise mit sieben Nachbarn, Kosinusdistanz und gleichen Stimmgewichten erzeugt.

In [ ]:
import csv

output_path = data_dir / 'my_test_predictions.csv'
prediction_rows = []
for path, actual, predicted in zip(test_paths, test_labels, test_predictions):
    prediction_rows.append({
        'file_name': path.name,
        'actual': actual,
        'predicted': predicted,
    })

In [ ]:
with output_path.open('w', encoding='utf-8', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=['file_name', 'actual', 'predicted'])
    writer.writeheader()
    writer.writerows(prediction_rows)

print(*output_path.read_text(encoding='utf-8').splitlines()[:4], sep='\n')